In [ ]:
import pandas as pd

# Define `df` by loading your reproduced experiment table here.


In [ ]:
cols_to_plot = [
    'sweep_pair.num_heads',
    'sweep_pair.num_mkeys',
    'sweep_pair.num_nkeys',
    'sweep_pair.dict_size',
    'explained_variance',
    'parameters_count',
    'accum_num_flops',
    'n_tokens',
    'performance/recovery_from_zero',
    'performance/recovery_from_mean'
]

sae_type_to_name = {
    "topk": "TopK",
    "mul_fractal_topk": "KronSAE",
    "jumprelu": "JumpReLU",
    "mul_fractal_jumprelu": "KronSAE JumpReLU",
    "topk_matryoshka": "Matryoshka",
    "kron_topk_matryoshka": "KronSAE Matryoshka",
    "kron_switch": "KronSAE Switch",
    "switch": "Switch",
}

def map_to_model(txt):
    if "1.5B" in txt:
        return "Qwen-2.5 1.5B"
    if "2b" in txt:
        return "Gemma-2 2B"


def map_to_ref(txt):
    if "500m" in txt:
        return "500m"
    if "1000m" in txt:
        return "1000m"
    if "2000m" in txt:
        return "2000m"


logic = []
topk = []
for i, row in df.iterrows():
    # if 'exp' in row['Name'] and row["topk"] == 50: #and row['Sweep'] in mapping.keys():
    model_name =  map_to_model(row['model_name'].split("-")[-1])
    model_size =  row['model_name'].split("-")[-1]
    ref_tokens = map_to_ref(row['run_name']) # row['Name'].split("_")[-2]

    sae_name = sae_type_to_name[row['sae_type']]

    group_size = ""
    if "Matryoshka" in sae_name:
        group_size = f'-{row["group_sizes"][1:-1].replace(" ", "").split(",")[0]}'
    # print()
    entry = {
        'Name': row['run_name'],
        'H': int(row['num_heads']),
        'M': int(row['num_mkeys']),
        'N': int(row['num_nkeys']),
        'F': int(row['dict_size']),
        'Layer': int(row['layer']),
        'SAE': sae_name+group_size,
        'Explained Variance': float(row['explained_variance']),
        'MSE': float(row['l2_loss']),
        'L0': int(row['topk']),
        'CE_from_zero': float(row['performance/recovery_from_zero']),
        'CE_from_mean': float(row['performance/recovery_from_mean']),
        'Parameters': float(row['parameters_count']),
        'FLOPS': float(row['accum_num_flops']),
        'Tokens': float(row['n_tokens']),
        'Reference Tokens': ref_tokens,
        'Model size': model_size,
        'Model name': model_name
    }
    
    if 'mul_fractal' in row['run_name']:
        entry['Experts'] = entry['H'] * entry['M']
        logic.append(entry)
    else:
        topk.append(entry)

logic = pd.DataFrame(logic)
topk = pd.DataFrame(topk)
data = pd.concat([logic, topk]).reset_index(drop=True)
data.head(), data.shape

In [ ]:
data[(data["L0"] == 8) & (data["F"] == 65535)]

In [ ]:
data["Explained Variance"].min()

In [ ]:
import colorsys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

for layer in data["Layer"].unique():
    fig, axs = plt.subplots(2, 3, figsize=(14, 7), dpi=300, sharex=True, sharey="row")
    tab10 = plt.cm.tab10.colors  # Get base colors: [(0.12, 0.47, 0.71, 1), ...]
    
    
    sae_pairs = [
        ("TopK", "KronSAE"),
        # ("Matryoshka", "KronSAE Matryoshka"),
        # ("Switch", "KronSAE Switch"),
    ]
    
    sae_map = {
        "KronSAE" : "KronSAE",
        "KronSAE Matryoshka": "KronSAE + Matryoshka",
        "KronSAE Switch" : "KronSAE + Switch",
    }
    
    l0s = list(data["L0"].unique())
    lightness_vals = np.linspace(0.8, 0.4, 4)
    
    for i, model in enumerate(list(data["Model name"].unique())[::-1]):
        baseline, kron_sae_type = sae_pairs[0]
        for j, d in enumerate(list(data["F"].unique())[::1]):
            subset = data[(data["Model name"] == model)] # & (data["Explained Variance"] > 0.3)]
            # subset = data[(data["F"] == d)]
            subset = subset[(subset["F"] == d) & (subset["Layer"] == layer)]
            subset = subset[subset["L0"].isin([8, 16, 32, 64])]
            # subset = subset[subset["M"].isin(l0s)]
            
            # Process KronSAE data
            # kron = subset[(subset["SAE"] == kron) & (subset["M"] <= 16)].copy()
            kron = subset[(subset["SAE"] == kron_sae_type)].copy()
            # print(kron)
            if not kron.empty:
                kron['M_float'] = kron['M'].astype(float)
                m_vals = sorted(kron['M_float'].unique())
                kron['M'] = kron['M_float'].apply(lambda x: str(int(x)))  # Clean integer strings
                
                # Generate shades: lighter (small M) → darker (large M)
                base_rgb = tab10[i][:3]  # Base color (R,G,B) for this dict size
                  # Light to dark
                kron_colors = []
                for l_val in lightness_vals:
                    h, l, s = colorsys.rgb_to_hls(*base_rgb)
                    new_rgb = colorsys.hls_to_rgb(h, l_val, s)
                    kron_colors.append(new_rgb)
            else:
                m_vals = []
                kron_colors = []
            
            # Process TopK (single bar)
            topk = subset[subset["SAE"] == baseline].copy()
            topk['M'] = baseline
            topk_color = (0.267, 0.267, 0.267)  # #444444 (dark gray)
            
            # Combine datasets
            combined = pd.concat([kron, topk])
            m_order = [str(int(m)) for m in m_vals] + [baseline]  # Numeric sort for KronSAE
            print(m_order)
            palette = kron_colors + [topk_color]  # KronSAE shades + TopK color
            print(palette)
            
            # Plot with custom palette
            sns.barplot(
                combined,
                x="L0",
                y="Explained Variance",
                hue="M",
                estimator="max",
                ax=axs[i, j],
                legend=j==2,
                # order="500m, 1000m",
                hue_order=m_order,
                palette=palette if j != 0 else palette[:-2] + [palette[-1]],
                # order=["500m", "1000m"],
                dodge=True
            )
            # if i==0:
            # axs[i, j].set_title(f"Model: {model}")
            if i==0:
                # axs[i, j].set_title(f"{baseline} / {sae_map[kron_sae_type]}", fontsize=14)
                axs[i, j].set_title(f"F = {d}", fontsize=14)
                axs[i, j].set_ylim(subset["Explained Variance"].min() - 0.1, subset["Explained Variance"].max() + 0.1)
            if i==1:
                axs[i, j].set_xlabel(r'$\ell_0$', fontsize=16)
                axs[i, j].set_ylim(subset["Explained Variance"].min() - 0.1, subset["Explained Variance"].max() + 0.1)
            if j==0:
                axs[i, j].set_ylabel('')
            # if j==1:
            axs[i, j].grid(alpha=0.35, axis='y', linestyle='--')
            axs[i, j].grid(alpha=0.35, axis='x', linestyle='--')
    
            if j == 2:
                handles, labels = axs[i, j].get_legend_handles_labels()
                axs[i, j].legend(
                    handles= [handles[i] for i in range(5)],  # Smallest M + TopK
                    labels=[f"m={labels[i]}" for i in range(4)]+[ "Baseline"],
                    loc="upper left",
                    frameon=True,
                    framealpha=0.9,
                    facecolor='white',
                    fontsize=11
                )
                # axs[i, j].set_yticks([])
                ax2 = axs[i, j].twinx()
                ax2.set_ylabel(model, fontsize=14)
                ax2.set_yticks([])
                ax2.yaxis.set_label_position('right') 
    
    
    fig.supylabel('Explained Variance', fontsize=16)
    
    plt.tight_layout()
    plt.savefig(f"l0_big_transpose_layer_{layer}.pdf")
    print(f"l0_big_transpose_layer_{layer}.pdf")
    plt.show()

In [ ]:

plot_df = data.copy().copy()
plot_df = plot_df[plot_df["L0"].isin([8, 16, 32, 64])].copy()
plot_df["Forward FLOPs"] = plot_df.apply(row_flops, axis=1)
plot_df["Total FLOPs"] = plot_df["Forward FLOPs"] * pd.to_numeric(plot_df["Tokens"])
plot_df["M_label"] = plot_df["M"].astype(float).astype(int).astype(str)
plot_df.loc[plot_df["SAE"].eq("TopK"), "M_label"] = "Baseline"
baseline = (
    plot_df[plot_df["SAE"].eq("TopK")]
    .groupby(["Model name", "F", "L0"], as_index=False)
    .agg({
        "Total FLOPs": "max",
        "Explained Variance": "max",
    })
    .rename(columns={
        "Total FLOPs": "TopK FLOPs",
        "Explained Variance": "TopK EV",
    })
)

plot_df = plot_df.merge(baseline, on=["Model name", "F", "L0"], how="left")
plot_df["FLOPs Change (%)"] = 100 * (plot_df["Total FLOPs"] / plot_df["TopK FLOPs"] - 1)
plot_df["EV Change (%)"] = 100 * (plot_df["Explained Variance"] / plot_df["TopK EV"] - 1)

fig, axs = plt.subplots(2, 3, figsize=(14, 7), dpi=300)

for i, model in enumerate(models):
    for j, F in enumerate(Fs):
        ax = axs[i, j]
        subset = plot_df[(plot_df["Model name"] == model) & (plot_df["F"] == F)].copy()
        # subset = subset[subset["L0"] == 64].copy()

        m_order = sorted(
            [x for x in subset["M_label"].dropna().unique() if x != "Baseline"],
            key=lambda x: int(x),
        ) + ["Baseline"]

        palette = sns.color_palette("Blues" if i == 0 else "Oranges", len(m_order) + 2)[1:len(m_order)]
        palette = dict(zip(m_order, palette))
        palette["Baseline"] = (0.267, 0.267, 0.267)

        sns.scatterplot(
            data=subset,
            x="FLOPs Change (%)",
            y="EV Change (%)",
            hue="M_label",
            # style="L0",
            hue_order=m_order,
            palette=palette,
            s=70,
            alpha=0.85,
            ax=ax,
            legend=j == 2,
        )

        def upper_convex_front(df, x="FLOPs Change (%)", y="EV Change (%)", eps=0.3):
            pts = (
                df[[x, y]]
                .dropna()
                .sort_values(x)
                .reset_index(drop=True)
            )
        
            collapsed = []
            group = []
        
            for _, row in pts.iterrows():
                if not group or abs(row[x] - group[0][x]) <= eps:
                    group.append(row)
                else:
                    collapsed.append(max(group, key=lambda r: r[y]))
                    group = [row]
        
            if group:
                collapsed.append(max(group, key=lambda r: r[y]))
        
            pts = pd.DataFrame(collapsed)[[x, y]].sort_values(x).to_numpy()
        
            hull = []
            for p in pts:
                while len(hull) >= 2:
                    o, a = hull[-2], hull[-1]
                    cross = (a[0] - o[0]) * (p[1] - o[1]) - (a[1] - o[1]) * (p[0] - o[0])
                    if cross >= 0:
                        hull.pop()
                    else:
                        break
                hull.append(p)
        
            return pd.DataFrame(hull, columns=[x, y])

        
                
        kron = subset[subset["SAE"].astype(str).str.contains("KronSAE", na=False)].copy()
        front = upper_convex_front(kron)
        
        sns.lineplot(
            data=front,
            x="FLOPs Change (%)",
            y="EV Change (%)",
            color="red",
            linewidth=1.5,
            marker=".",
            alpha=0.75,
            ax=ax,
            legend=False,
        )
        from matplotlib.ticker import PercentFormatter

        ax.xaxis.set_major_formatter(PercentFormatter(xmax=100))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))

        ax.axhline(0, color="black", linewidth=1, alpha=0.35)
        ax.axvline(0, color="black", linewidth=1, alpha=0.35)
        ax.grid(alpha=0.35, axis="both", linestyle="--")
        ax.set_xlabel("FLOPs change vs TopK (%)" if i == 1 else "")
        ax.set_ylabel("")

        if i == 0:
            ax.set_title(f"F = {int(F)}", fontsize=14)

        # if j == 2:
        #     ax2 = ax.twinx()
        #     ax2.set_ylabel(model, fontsize=14)
        #     ax2.set_yticks([])
        if j == 2:
            handles, labels = axs[i, j].get_legend_handles_labels()
            axs[i, j].legend(
                handles= [handles[i] for i in range(5)],  # Smallest M + TopK
                labels=[f"m={labels[i]}" for i in range(4)]+[ "Baseline"],
                loc="lower right",
                frameon=True,
                framealpha=0.9,
                facecolor='white',
                fontsize=11
            )
            # axs[i, j].set_yticks([])
            ax2 = axs[i, j].twinx()
            ax2.set_ylabel(model, fontsize=14)
            ax2.set_yticks([])
            ax2.yaxis.set_label_position('right') 

fig.supylabel("Explained Variance change vs TopK (%)", fontsize=16)
plt.tight_layout()
plt.savefig("flops_vs_ev_percent_change_kron_pareto.pdf")
plt.show()


In [ ]:
import colorsys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

for layer in data["Layer"].unique():
    fig, axs = plt.subplots(2, 3, figsize=(14, 7), dpi=300, sharex=True, sharey=True)
    tab10 = plt.cm.tab10.colors  # Get base colors: [(0.12, 0.47, 0.71, 1), ...]
    
    
    sae_pairs = [
        ("TopK", "KronSAE"),
        # ("Matryoshka", "KronSAE Matryoshka"),
        # ("Switch", "KronSAE Switch"),
    ]
    
    sae_map = {
        "KronSAE" : "KronSAE",
        "KronSAE Matryoshka": "KronSAE + Matryoshka",
        "KronSAE Switch" : "KronSAE + Switch",
    }
    
    l0s = list(data["L0"].unique())
    lightness_vals = np.linspace(0.8, 0.4, 4)
    
    for i, model in enumerate(list(data["Model name"].unique())[::-1]):
        baseline, kron_sae_type = sae_pairs[0]
        for j, d in enumerate(list(data["F"].unique())[::1]):
            # subset = data[(data["Model name"] == model)] # & (data["Explained Variance"] > 0.3)]
            # subset = data[(data["F"] == d)]
            subset = data[(data["F"] == d) & (data["Layer"] == layer)]
            subset = subset[subset["L0"].isin([8, 16, 32, 64])]
            subset = subset[subset["N"] <=16]
            # subset = subset[subset["M"].isin(l0s)]
            
            # Process KronSAE data
            # kron = subset[(subset["SAE"] == kron) & (subset["M"] <= 16)].copy()
            kron = subset[(subset["SAE"] == kron_sae_type)].copy()
            # print(kron)
            if not kron.empty:
                kron['M_float'] = kron['M'].astype(float)
                m_vals = sorted(kron['M_float'].unique())
                kron['M'] = kron['M_float'].apply(lambda x: str(int(x)))  # Clean integer strings
                
                # Generate shades: lighter (small M) → darker (large M)
                base_rgb = tab10[i][:3]  # Base color (R,G,B) for this dict size
                  # Light to dark
                kron_colors = []
                for l_val in lightness_vals:
                    h, l, s = colorsys.rgb_to_hls(*base_rgb)
                    new_rgb = colorsys.hls_to_rgb(h, l_val, s)
                    kron_colors.append(new_rgb)
            else:
                m_vals = []
                kron_colors = []
            
            # Process TopK (single bar)
            topk = subset[subset["SAE"] == baseline].copy()
            topk['M'] = baseline
            topk_color = (0.267, 0.267, 0.267)  # #444444 (dark gray)
            
            # Combine datasets
            combined = pd.concat([kron, topk])
            m_order = [str(int(m)) for m in m_vals] + [baseline]  # Numeric sort for KronSAE
            print(m_order)
            palette = kron_colors + [topk_color]  # KronSAE shades + TopK color
            print(palette)
            
            # Plot with custom palette
            sns.barplot(
                combined,
                x="L0",
                y="MSE",
                hue="M",
                estimator="min",
                ax=axs[i, j],
                legend=j==2,
                # order="500m, 1000m",
                hue_order=m_order,
                palette=palette if j != 0 else palette[:-2] + [palette[-1]],
                # order=["500m", "1000m"],
                dodge=True
            )
            # if i==0:
            # axs[i, j].set_ylim(0.6, 0.94)
            # axs[i, j].set_title(f"Model: {model}")
            if i==0:
                # axs[i, j].set_title(f"{baseline} / {sae_map[kron_sae_type]}", fontsize=14)
                axs[i, j].set_title(f"F = {d}", fontsize=14)
            if i==1:
                axs[i, j].set_xlabel(r'$\ell_0$', fontsize=16)
            if j==0:
                axs[i, j].set_ylabel('')
            axs[i, j].grid(alpha=0.35, axis='y', linestyle='--')
            axs[i, j].grid(alpha=0.35, axis='x', linestyle='--')
    
            if j == 2:
                handles, labels = axs[i, j].get_legend_handles_labels()
                axs[i, j].legend(
                    handles= [handles[i] for i in range(5)],  # Smallest M + TopK
                    labels=[f"m={labels[i]}" for i in range(4)]+[ "Baseline"],
                    loc="upper left",
                    frameon=True,
                    framealpha=0.9,
                    facecolor='white',
                    fontsize=11
                )
                # axs[i, j].set_yticks([])
                ax2 = axs[i, j].twinx()
                ax2.set_ylabel(model, fontsize=14)
                ax2.set_yticks([])
                ax2.yaxis.set_label_position('right') 
    
    
    fig.supylabel('MSE', fontsize=16)
    
    plt.tight_layout()
    plt.savefig(f"l0_big_transpose_layer_{layer}_mse.pdf")
    plt.show()